# NN_12 — Ablation: DNN-1024 com entrada z_eq

**Questão central**: mantendo a *arquitetura* DNN (256→128→64→sigmoid), mas substituindo
as 4 features extraídas pelo vetor bruto z_eq ∈ R¹⁰²⁴, é possível obter um resultado bom
de PD vs SNR?

**Motivação do ablation**: A comparação 4-feat DNN → 1D CNN confundiu dois efeitos:
1. Mudança do vetor de entrada (4 features → 1024-dim z_eq)  
2. Mudança de arquitetura (DNN → CNN com compartilhamento de pesos)

Este notebook isola o efeito da arquitetura: mantemos o DNN idêntico ao `NN_02b`
(mesmos hiperparâmetros, mesmos callbacks) e apenas trocamos o input de 4 para 1024.

| Modelo | Entrada | Arquitetura | Compartilhamento |
|--------|---------|-------------|------------------|
| 4-feat DNN | [\|τ\|, ĥ, SNR, E] ∈ R⁴ | Dense 256→128→64 | Não |
| **DNN-1024** | z_eq ∈ R¹⁰²⁴ (flattened) | Dense 256→128→64 | **Não** |
| 1D CNN | z_eq ∈ R¹⁰²⁴×¹ | Conv32→Conv64→Conv128→GAP→Dense64 | **Sim** |

**Input**: `z_eq = y/h − ρ_s·s` (sinal residual após remoção da mensagem)  
  - H1: z_eq = ρ_t·t + w/h  (TAG + ruído)  
  - H0: z_eq ≈ w/h          (ruído puro, pois 1−ρ_s ≈ 0.008)

**Threshold**: D3F Gaussiano (Braca 2022, Eq. 20): τ*(α) = μ_H0 + Q⁻¹(1−α)·σ_H0,  
com Q⁻¹(1−10⁻⁷) ≈ 5.199

Modelo salvo em `model_dnn1024_ablation.keras`.

In [ ]:
# ==============================================================================
# 1. IMPORTS & GPU CONFIGURATION
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
from pathlib import Path
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import gc
import psutil
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

print(f"TensorFlow : {tf.__version__}")

# ── GPU: enable memory growth ──────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs found : {gpus}")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    USE_GPU = True
    policy = keras.mixed_precision.Policy('mixed_float16')
    keras.mixed_precision.set_global_policy(policy)
    print(f"Memory growth enabled  |  Mixed precision: {policy.name}")
else:
    USE_GPU = False
    print("No GPU detected — running on CPU.")

tf.config.optimizer.set_jit(True)
print("XLA JIT : enabled")

np.random.seed(42)
tf.random.set_seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
project_root       = Path.cwd().parent
results_dir        = project_root / "results"
data_dir           = results_dir / "data"
models_dir         = results_dir / "models"
visualizations_dir = results_dir / "visualizations"
models_dir.mkdir(parents=True, exist_ok=True)
visualizations_dir.mkdir(parents=True, exist_ok=True)

print(f"Data dir   : {data_dir}")
print(f"Models dir : {models_dir}")

In [ ]:
# ==============================================================================
# 2. LOAD DATASET — z_eq flattened to (N, 1024)
# ==============================================================================
# z_eq = y/h - rho_s * msg  (residual após remoção da mensagem conhecida)
#
#   H1: z_eq = rho_t * tag + w/h   — TAG imersa em ruído
#   H0: z_eq ≈ w/h                 — ruído puro
#
# DNN-1024 recebe z_eq como vetor plano: Input(1024) sem redimensionamento.
# Isso contrasta com a 1D CNN que recebe z_eq como (1024, 1) e aplica
# Conv1D com compartilhamento de pesos.

dataset_path = data_dir / "dataset_cnn_yeq_0_30dB.h5"

if not dataset_path.exists():
    raise FileNotFoundError(
        f"{dataset_path} not found.\nRun NN_01_DataGeneration.ipynb first."
    )

with h5py.File(str(dataset_path), 'r') as f:
    if 'train/z_eq' not in f:
        raise KeyError(
            "z_eq não encontrado no dataset.\n"
            "Re-execute NN_01_DataGeneration.ipynb para regenerar o dataset com z_eq."
        )

    L_FIXED = int(f.attrs['L_FIXED'])
    RHO_S   = float(f.attrs.get('RHO_S', 0.992472))
    RHO_T   = float(f.attrs.get('RHO_T', 0.212132))

    # Flat input (N, 1024) — sem reshape para Conv1D
    X_train_raw = f['train/z_eq'][:].astype(np.float32)   # (N, 1024) — will be freed after normalization
    y_train     = f['train/y'][:].astype(np.float32)
    snr_train   = f['train/snr'][:]

    X_val_raw   = f['val/z_eq'][:].astype(np.float32)
    y_val       = f['val/y'][:].astype(np.float32)
    snr_val     = f['val/snr'][:]

    X_test_raw  = f['test/z_eq'][:].astype(np.float32)
    y_test      = f['test/y'][:].astype(np.float32)
    snr_test    = f['test/snr'][:]

    # Carregar tau_eq para comparação com correlator clássico
    TAU_val  = f['val/tau_eq'][:].astype(np.float32)
    TAU_test = f['test/tau_eq'][:].astype(np.float32)

print(f"Parâmetros: RHO_T={RHO_T:.4f}  RHO_S={RHO_S:.4f}  L={L_FIXED}")
print(f"Train : {X_train_raw.shape}  H0={(y_train==0).sum()}  H1={(y_train==1).sum()}")
print(f"Val   : {X_val_raw.shape}")
print(f"Test  : {X_test_raw.shape}")

# Validação: H1 deve ter maior variância que H0 em alto SNR
print("\nValidação z_eq (variância média — H1 > H0 em alto SNR é esperado):")
for snr_lo in [0, 15, 25]:
    m1 = (y_test == 1) & (snr_test >= snr_lo) & (snr_test < snr_lo + 5)
    m0 = (y_test == 0) & (snr_test >= snr_lo) & (snr_test < snr_lo + 5)
    if m1.sum() > 50:
        v1 = X_test_raw[m1].var(axis=1).mean()
        v0 = X_test_raw[m0].var(axis=1).mean()
        print(f"  SNR [{snr_lo},{snr_lo+5}] dB: H1={v1:.5f}  H0={v0:.5f}  ratio={v1/(v0+1e-10):.3f}")

In [ ]:
# ==============================================================================
# 3. NORMALIZAÇÃO GLOBAL (z-score fit no treino)
# ==============================================================================
# Para o DNN-1024, usamos normalização global (escalar único) em vez de
# per-feature, pois todos os 1024 elementos são da mesma grandeza física (z_eq).
#
# Alternativa descartada: per-sample standardization removeria diferenças
# de amplitude entre SNRs — que são informativas para o detector.

sc_mean = float(X_train_raw.mean())
sc_std  = float(X_train_raw.std()) + 1e-8

X_train = ((X_train_raw - sc_mean) / sc_std).astype(np.float32)
del X_train_raw; gc.collect()  # free ~1.1 GB

X_val   = ((X_val_raw   - sc_mean) / sc_std).astype(np.float32)
del X_val_raw; gc.collect()

X_test  = ((X_test_raw  - sc_mean) / sc_std).astype(np.float32)
del X_test_raw; gc.collect()

import psutil
print(f"RAM after normalization: {psutil.virtual_memory().available/1e9:.2f} GB free")

print(f"Scaler: mean={sc_mean:.6f}  std={sc_std:.6f}")
print(f"X_train normalizado: mean={X_train.mean():.4f}  std={X_train.std():.4f}")
print(f"X_train shape: {X_train.shape}  (input_dim do DNN = {X_train.shape[1]})")

# Salvar scaler para reprodutibilidade
scaler_path = models_dir / "scaler_dnn1024_ablation.json"
with open(str(scaler_path), 'w') as f_sc:
    json.dump({'mean': sc_mean, 'std': sc_std,
               'input': 'z_eq (flattened 1024-dim)',
               'dataset': 'dataset_cnn_yeq_0_30dB.h5'}, f_sc, indent=2)
print(f"Scaler salvo → {scaler_path}")

In [ ]:
# ==============================================================================
# 4. ARQUITETURA DNN-1024 — mesma topologia do NN_02b, input_dim=1024
# ==============================================================================
# Idêntico ao build_dnn_4feat() do NN_02b, exceto input_dim=1024.
# Isso garante que qualquer diferença de desempenho vs 4-feat DNN se deva
# exclusivamente à representação do input, não à arquitetura.
#
# Parâmetros:
#   Layer 1: Dense(256)  → 1024×256 + 256 bias + BN(4×256) = 263,680
#   Layer 2: Dense(128)  → 256×128  + 128 bias + BN(4×128) =  33,280
#   Layer 3: Dense(64)   → 128×64   + 64  bias             =   8,256
#   Output:  Dense(1)    → 64×1     + 1   bias             =      65
#   Total ≈ 305k params (vs ~44k do 4-feat DNN e ~69k da 1D CNN)

def build_dnn_1024(input_dim=1024, dropout=(0.3, 0.3, 0.2)):
    inp = layers.Input(shape=(input_dim,), name='z_eq_input')

    x = layers.Dense(256, kernel_initializer='he_uniform')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout[0])(x)

    x = layers.Dense(128, kernel_initializer='he_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout[1])(x)

    x = layers.Dense(64, kernel_initializer='he_uniform')(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout[2])(x)

    # float32 explícito — necessário quando mixed_float16 está ativo
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='P_H1')(x)
    return keras.Model(inputs=inp, outputs=out, name='DNN_1024_Ablation')


if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
    with strategy.scope():
        model = build_dnn_1024(input_dim=L_FIXED)
        model.compile(
            optimizer=Adam(learning_rate=1e-3),
            loss=BinaryCrossentropy(),
            metrics=['accuracy', keras.metrics.AUC(name='auc')]
        )
else:
    model = build_dnn_1024(input_dim=L_FIXED)
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=BinaryCrossentropy(),
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )

model.summary()
print(f"\nTotal trainable params : {model.count_params():,}")
print(f"\nComparação de parâmetros:")
print(f"  4-feat DNN (NN_02b)   : ~44k  params, input=4")
print(f"  DNN-1024  (este)      : ~{model.count_params()//1000}k params, input=1024")
print(f"  1D CNN    (NN_02)     : ~69k  params, input=(1024,1), Conv+GAP")

In [ ]:
# ==============================================================================
# 5. CALLBACKS — idênticos ao NN_02b (4-feat DNN) para comparação justa
# ==============================================================================

model_path = str(models_dir / 'model_dnn1024_ablation.keras')

cbs = [
    callbacks.EarlyStopping(
        monitor='val_auc', mode='max', patience=20,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        model_path, monitor='val_auc', mode='max',
        save_best_only=True, verbose=0
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=8,
        min_lr=1e-6, verbose=1
    ),
]

print(f"Checkpoint → {model_path}")
print(f"EarlyStopping: patience=20, monitor=val_auc")
print(f"ReduceLROnPlateau: factor=0.5, patience=8, min_lr=1e-6")

In [ ]:
# ==============================================================================
# 6. TREINAMENTO
# ==============================================================================
# Mesmos hiperparâmetros do NN_02b:
#   batch_size=512, max_epochs=150, Adam lr=1e-3
#
# DNN-1024 é mais pesado que o 4-feat (~7× mais params), mas ainda leve
# comparado à 1D CNN em complexidade computacional por batch.

BATCH_SIZE = 512 if USE_GPU else 256

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=BATCH_SIZE,
    callbacks=cbs,
    verbose=1
)

best_val_auc = max(history.history['val_auc'])
print(f"\nBest val AUC  : {best_val_auc:.5f}")
print(f"Best val loss : {min(history.history['val_loss']):.5f}")
print(f"Epochs run    : {len(history.history['loss'])}")

In [ ]:
# ==============================================================================
# 7. INFERÊNCIA
# ==============================================================================

p_val  = model.predict(X_val,  batch_size=1024, verbose=0).flatten()
p_test = model.predict(X_test, batch_size=1024, verbose=0).flatten()

auc_val  = roc_auc_score(y_val,  p_val)
auc_test = roc_auc_score(y_test, p_test)

print(f"Val  AUC : {auc_val:.5f}")
print(f"Test AUC : {auc_test:.5f}")

# Distribuição dos scores por hipótese
print("\nDistribuição P(H1) — val set:")
for lbl, name in [(0, 'H0'), (1, 'H1')]:
    s = p_val[y_val == lbl]
    print(f"  {name}: mean={s.mean():.4f}  std={s.std():.4f}  "
          f"min={s.min():.4f}  max={s.max():.4f}")

In [ ]:
# ==============================================================================
# 8. D3F THRESHOLD CALIBRATION — α = 10⁻⁷ (Braca 2022, Eq. 20)
# ==============================================================================
# τ*(α) = μ_H0 + Q⁻¹(1−α)·σ_H0
# Q⁻¹(1−10⁻⁷) ≈ 5.199
#
# Calibrado POR BIN DE SNR no conjunto de validação (H0):
# cada bin tem um limiar adaptado às estatísticas locais do score H0.

SNR_POINTS = np.arange(0, 31, 5)
HALF_BIN   = 2.5
ALPHA      = 1e-7
q_inv      = norm.ppf(1 - ALPHA)
print(f"Q⁻¹(1 − 10⁻⁷) = {q_inv:.4f}")

# Máscaras vetorizadas para H0 no val set
is_h0_val   = (y_val == 0)
in_bin_lo_v = snr_val[:, None] >= SNR_POINTS[None, :] - HALF_BIN
in_bin_hi_v = snr_val[:, None] <  SNR_POINTS[None, :] + HALF_BIN
masks_h0_val = (is_h0_val[:, None] & in_bin_lo_v & in_bin_hi_v).T  # (n_bins, N_val)

thresholds_dnn1024 = {}
thresholds_tau     = {}

print(f"\n{'SNR':>5}  {'N_H0_val':>9}  {'μ_H0':>9}  {'σ_H0':>9}  {'τ*_DNN1024':>12}  {'τ*_Classical':>14}")
print("-" * 70)

for i, snr in enumerate(SNR_POINTS):
    mask = masks_h0_val[i]
    n_h0 = int(mask.sum())

    if n_h0 < 30:
        thresholds_dnn1024[int(snr)] = np.nan
        thresholds_tau[int(snr)]     = np.nan
        print(f"{snr:>5}  {n_h0:>9}  {'N/A':>9}")
        continue

    # DNN-1024 D3F threshold
    s_dnn  = p_val[mask]
    mu_d, sig_d = float(s_dnn.mean()), float(s_dnn.std())
    tau_dnn = float(np.clip(mu_d + q_inv * sig_d, None, 1.0))

    # Classical correlator D3F threshold
    s_tau  = TAU_val[mask]
    mu_t, sig_t = float(s_tau.mean()), float(s_tau.std())
    tau_tau = float(mu_t + q_inv * sig_t)

    thresholds_dnn1024[int(snr)] = tau_dnn
    thresholds_tau[int(snr)]     = tau_tau

    print(f"{snr:>5}  {n_h0:>9}  {mu_d:>9.5f}  {sig_d:>9.5f}"
          f"  {tau_dnn:>12.6f}  {tau_tau:>14.4f}")

In [ ]:
# ==============================================================================
# 9. PD vs SNR — DNN-1024 e Classical
# ==============================================================================

is_h1_test   = (y_test == 1)
in_bin_lo_t  = snr_test[:, None] >= SNR_POINTS[None, :] - HALF_BIN
in_bin_hi_t  = snr_test[:, None] <  SNR_POINTS[None, :] + HALF_BIN
masks_h1_test = (is_h1_test[:, None] & in_bin_lo_t & in_bin_hi_t).T  # (n_bins, N_test)

pd_dnn1024   = {}
pd_classical = {}
n_h1_per_bin = {}

print(f"\n{'SNR':>5}  {'N_H1':>7}  {'PD_DNN1024':>12}  {'PD_Classical':>14}")
print("-" * 45)

for i, snr in enumerate(SNR_POINTS):
    mask = masks_h1_test[i]
    n_h1 = int(mask.sum())
    n_h1_per_bin[int(snr)] = n_h1

    if n_h1 < 5 or np.isnan(thresholds_dnn1024.get(int(snr), np.nan)):
        pd_dnn1024[int(snr)]   = np.nan
        pd_classical[int(snr)] = np.nan
        print(f"{snr:>5}  {n_h1:>7}  {'N/A':>12}")
        continue

    p_h1_dnn = p_test[mask]
    tau_h1   = TAU_test[mask]

    pd_d = float((p_h1_dnn >= thresholds_dnn1024[int(snr)]).mean())
    pd_t = float((tau_h1   >= thresholds_tau[int(snr)]).mean())

    pd_dnn1024[int(snr)]   = pd_d
    pd_classical[int(snr)] = pd_t

    bar_d = '#' * int(pd_d * 30)
    bar_t = '#' * int(pd_t * 30)
    print(f"{snr:>5}  {n_h1:>7}  {pd_d:>12.4f}  {pd_t:>14.4f}")

In [ ]:
# ==============================================================================
# 10. CARREGAR RESULTADOS EXISTENTES PARA COMPARAÇÃO
# ==============================================================================
# Carrega: 1D CNN (de nn09_logit_d3f.json, avaliado com z_eq) e
#          4-feat DNN sigmoid (de nn09_logit_d3f.json, resultado referência).

nn09_path = data_dir / "nn09_logit_d3f.json"
pd_cnn    = {}
pd_4feat  = {}

if nn09_path.exists():
    with open(str(nn09_path)) as f_in:
        nn09 = json.load(f_in)

    # 1D CNN sigmoid (avaliado com z_eq em NN_09)
    for k, v in nn09.get('cnn_sigmoid', {}).items():
        pd_cnn[int(k)] = v if v is not None else float('nan')

    # 4-feat DNN sigmoid (resultado referência — AUC 0.9998)
    for k, v in nn09.get('4feat_sigmoid', {}).items():
        pd_4feat[int(k)] = v if v is not None else float('nan')

    print(f"Resultados carregados de {nn09_path.name}")
    print(f"  1D CNN sigmoid  : {pd_cnn}")
    print(f"  4-feat DNN sig  : {pd_4feat}")
else:
    # Fallback: figura2 (CNN com y_eq — menos preciso)
    fig2_path = data_dir / "figure2_pd_vs_snr_gpu.json"
    if fig2_path.exists():
        with open(str(fig2_path)) as f_in:
            fig2 = json.load(f_in)
        for k, v in fig2['method_cnn']['PD_vs_SNR'].items():
            pd_cnn[int(k)] = v if v is not None else float('nan')
        print(f"Fallback: CNN carregado de {fig2_path.name}")
    else:
        print("WARNING: nn09_logit_d3f.json não encontrado. Execute NN_09 primeiro.")
        pd_cnn = {int(s): float('nan') for s in SNR_POINTS}
    pd_4feat = {int(s): float('nan') for s in SNR_POINTS}
    print("4-feat DNN: não disponível sem nn09_logit_d3f.json")


In [ ]:
# ==============================================================================
# 11. FIGURA — PD vs SNR: Ablation Study
# ==============================================================================

snr_arr      = np.array(SNR_POINTS, dtype=float)
pd_d1024_arr = np.array([pd_dnn1024.get(int(s), np.nan) for s in SNR_POINTS])
pd_class_arr = np.array([pd_classical.get(int(s), np.nan) for s in SNR_POINTS])
pd_cnn_arr   = np.array([pd_cnn.get(int(s), np.nan) for s in SNR_POINTS])
pd_4f_arr    = np.array([pd_4feat.get(int(s), np.nan) for s in SNR_POINTS])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    f"Ablation Study — Arquitetura DNN vs CNN  (α = {ALPHA:.0e}, D3F Gaussiano)",
    fontsize=13, fontweight='bold'
)

# ── Painel esquerdo: curvas PD vs SNR ────────────────────────────────────
ax = axes[0]

ax.plot(snr_arr, pd_class_arr, 'o-',  color='tomato',     lw=2.5, ms=8,
        markerfacecolor='white', markeredgewidth=2,
        label='Correlador Clássico (Xie 2021)')

if not np.all(np.isnan(pd_4f_arr)):
    ax.plot(snr_arr, pd_4f_arr, '^-', color='darkorange',  lw=2.5, ms=8,
            markerfacecolor='white', markeredgewidth=2,
            label='DNN 4-feat (NN_02b)')

ax.plot(snr_arr, pd_d1024_arr, 's--', color='mediumpurple', lw=2.5, ms=8,
        markerfacecolor='white', markeredgewidth=2,
        label='DNN-1024 z_eq (ablation)')

if not np.all(np.isnan(pd_cnn_arr)):
    ax.plot(snr_arr, pd_cnn_arr, 'D-.', color='steelblue',  lw=2.5, ms=8,
            markerfacecolor='white', markeredgewidth=2,
            label='1D CNN z_eq (NN_02 GPU)')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('Probabilidade de Detecção (PD)', fontsize=12)
ax.set_title('PD vs SNR — Comparação dos Modelos', fontsize=11)
ax.set_xlim(-1, 31)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(SNR_POINTS)
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)

# Caixa de informações
info = (
    f"BPSK | Rayleigh | L=1024\n"
    f"α = {ALPHA:.0e}  (D3F Gaussiano)\n"
    f"DNN-1024: {model.count_params():,} params\n"
    f"4-feat DNN: ~44k params\n"
    f"1D CNN:     ~69k params"
)
ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=8.5,
        va='top', ha='left',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

# ── Painel direito: curva de treinamento ──────────────────────────────────
ax2 = axes[1]
ax2.plot(history.history['loss'],     label='Train Loss', lw=2, color='steelblue')
ax2.plot(history.history['val_loss'], label='Val Loss',   lw=2, color='steelblue', ls='--')

ax2b = ax2.twinx()
ax2b.plot(history.history['auc'],     label='Train AUC', lw=2, color='tomato')
ax2b.plot(history.history['val_auc'], label='Val AUC',   lw=2, color='tomato', ls='--')
ax2b.set_ylabel('AUC', fontsize=11, color='tomato')
ax2b.set_ylim(0.4, 1.02)

ax2.set_xlabel('Época', fontsize=11)
ax2.set_ylabel('BCE Loss', fontsize=11)
ax2.set_title('Curva de Aprendizado — DNN-1024', fontsize=11)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')
ax2.grid(alpha=0.3)

plt.tight_layout()
fig_path = visualizations_dir / "NN12_ablation_dnn1024_pd_vs_snr.png"
plt.savefig(str(fig_path), dpi=150)
plt.show()
print(f"Figura salva → {fig_path}")

In [ ]:
# ==============================================================================
# 12. ANÁLISE FINAL DO ABLATION
# ==============================================================================
# Interpreta os resultados comparando DNN-1024 vs 4-feat DNN vs 1D CNN.
# Responde: "mantendo a arquitetura DNN, é possível obter bom PD vs SNR
# usando z_eq como input?"

print("=" * 65)
print("RESULTADO DO ABLATION STUDY")
print("=" * 65)
print(f"\nDNN-1024 (z_eq plano) — AUC: {auc_test:.5f}")

print(f"\nPD vs SNR (α={ALPHA:.0e}, D3F Gaussiano):")
print(f"{'SNR':>5}  {'DNN-1024':>10}  {'Classical':>11}  {'1D CNN':>10}")
print("-" * 43)
for snr in SNR_POINTS:
    d = pd_dnn1024.get(int(snr), np.nan)
    c = pd_classical.get(int(snr), np.nan)
    n = pd_cnn.get(int(snr), np.nan)
    d_s = f"{d:.4f}" if not np.isnan(d) else "N/A"
    c_s = f"{c:.4f}" if not np.isnan(c) else "N/A"
    n_s = f"{n:.4f}" if not np.isnan(n) else "N/A"
    print(f"{snr:>5}  {d_s:>10}  {c_s:>11}  {n_s:>10}")

print("\n--- Interpretação ---")
print("""
A DNN-1024 recebe z_eq como vetor PLANO (1024 entradas independentes).
Diferentemente da 1D CNN, não há compartilhamento de pesos entre posições:
a primeira camada densa precisa de 1024×256 pesos para aproximar o mesmo
filtro casado que a Conv1D aprende com apenas 32×15=480 pesos.

Hipóteses para interpretar o resultado:
  • DNN-1024 ≪ Classical → A arquitetura DNN sem compartilhamento de pesos
    não consegue explorar a estrutura sequencial de z_eq mesmo com ~305k params.
    O gargalo é inductive bias, não capacidade de parâmetros.
  • DNN-1024 ≈ 1D CNN → O compartilhamento de pesos da CNN não é essencial;
    o ganho da CNN vem do z_eq corrigido, não da arquitetura convolucional.
  • DNN-1024 entre 4-feat e CNN → Ambos os efeitos contribuem.
""")

# Salvar métricas
ablation_metrics = {
    'alpha': ALPHA,
    'auc_val':  float(auc_val),
    'auc_test': float(auc_test),
    'params': model.count_params(),
    'pd_vs_snr': {
        str(int(snr)): (None if np.isnan(pd_dnn1024.get(int(snr), np.nan))
                        else float(pd_dnn1024[int(snr)]))
        for snr in SNR_POINTS
    },
    'classical_pd_vs_snr': {
        str(int(snr)): (None if np.isnan(pd_classical.get(int(snr), np.nan))
                        else float(pd_classical[int(snr)]))
        for snr in SNR_POINTS
    },
    'n_h1_per_bin': n_h1_per_bin,
    'scaler': {'mean': sc_mean, 'std': sc_std},
    'model_path': model_path,
}

out_path = data_dir / "ablation_dnn1024_results.json"
with open(str(out_path), 'w') as f_out:
    json.dump(ablation_metrics, f_out, indent=2)

print(f"\nMétricas salvas → {out_path}")
print(f"Modelo salvo   → {model_path}")

## Resumo do Ablation

| Modelo | Input | Params | AUC | PD@20dB | Comentário |
|--------|-------|--------|-----|---------|------------|
| Classical (Xie 2021) | τ_eq (escalar) | — | — | — | Filtro casado |
| 4-feat DNN (NN_02b) | [\|τ\|,ĥ,SNR,E] ∈ R⁴ | ~44k | — | — | Features extraídas manualmente |
| **DNN-1024** (este) | z_eq ∈ R¹⁰²⁴ | ~305k | *run* | *run* | Sem compartilhamento |
| 1D CNN (NN_02 GPU) | z_eq ∈ R¹⁰²⁴×¹ | ~69k | — | — | Conv com peso compartilhado |

**Conclusão esperada**: Se a DNN-1024 tiver PD significativamente inferior à 1D CNN,
o inductive bias da convolução (invariância à translação + compartilhamento de pesos)
é o fator decisivo — não o tamanho do vetor de entrada.

Se a DNN-1024 tiver PD comparável à 1D CNN, a escolha de arquitetura é secundária
e o ganho vem principalmente da representação correta (z_eq vs y/h).